In [9]:

import sys
sys.path.append('../')
sys.path.append('../../')
import torch
from MINAR.ComputationGraph import ComputationGraph, Circuit
from model.CustomLosses import MultiplicativeLoss, JointLoss
import torch_geometric as pyg
import networkx as nx

import numpy as np
from model.MinAggGNN import MinAggGNN
import matplotlib.pyplot as plt


seeds = [0, 1, 2, 3, 4]
device = torch.device('cuda')
L = 2

test_data = torch.load('../data/test_data.pt', map_location=device, weights_only=False)
for data in test_data:
    data.x = torch.cat([data.x, data.x_bfs], 1)
test_loader = pyg.loader.DataLoader(test_data, batch_size = len(test_data))
test_loss = MultiplicativeLoss()
mse_loss = torch.nn.MSELoss()
bce_loss = torch.nn.BCEWithLogitsLoss()
criterion = JointLoss([mse_loss, bce_loss], [False, True], weight=torch.tensor([1.,25.], device=device))
num_reachable_test_nodes = sum([data.reachable.sum() for data in test_loader])

corrupted_data = torch.load('../data/test_data.pt', map_location=device, weights_only=False)
for data_corr in corrupted_data:
    data_corr.x = torch.cat([data_corr.x, data_corr.x_bfs], 1)
    data_corr.x = torch.zeros_like(data_corr.x, device=device)
    data_corr.x[0] = 0.
    data_corr.edge_attr = torch.zeros_like(data_corr.edge_attr, device=device)

def load_circuit(model):
    G = ComputationGraph(model)
    G.add_inputs({'edge_attr' : [1, model.convs[0].agg_mlp.lins[0].weight[:,-1]],
                  'input_self.0' : [3, model.convs[0].up_mlp.lins[0].weight[:,-2]],
                  'input_self.1' : [3, model.convs[0].up_mlp.lins[0].weight[:,-1]]})
    G.add_residual_connections({'edge_attr' : [5, model.convs[1].agg_mlp.lins[0].weight[:,-1].reshape(1,-1).cpu().detach()]})
    G.add_residual_connections({4 : [7, model.convs[1].up_mlp.lins[0].weight[:,-8:].T.cpu().detach()]})
    G.calculate_scores(test_data, corrupted_data, criterion, which = 'EAP-IG', steps=20)
    C = Circuit(model, G, K=11, key='EAP-IG')
    return C

In [11]:
model_losses = []
circuit_losses = []
circuit_ablated_losses = []

model_accs = []
circuit_accs = []
circuit_ablated_accs = []

model = MinAggGNN(2, 8, L, 2, edge_dim = 1)
for seed in seeds:
    state_dict = torch.load(f'../model_progress/parallel/seed_{seed}/model_final.pt')
    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)
    circuit = load_circuit(model)

    model_loss = 0
    circuit_loss = 0
    circuit_ablated_loss = 0

    model_acc = 0
    circuit_acc = 0
    circuit_ablated_acc = 0

    for data in test_loader:
        data = data.to(device)

        model_out = model(data.x, data.edge_index, edge_attr=data.edge_attr)
        circuit_out = circuit.forward(data)
        circuit_ablated_out = circuit.ablate_circuit(data)

        model_loss += float(test_loss(model_out[:,0][data.reachable].flatten(), data.y[data.reachable]).detach().item()) / num_reachable_test_nodes
        circuit_loss += float(test_loss(circuit_out[:,0][data.reachable].flatten(), data.y[data.reachable]).detach().item()) / num_reachable_test_nodes
        circuit_ablated_loss += test_loss(circuit_ablated_out[:,0][data.reachable].flatten(), data.y[data.reachable]).detach().item() / num_reachable_test_nodes

        model_acc += ((model_out[:,1].flatten() > 0) == data.reachable).sum().item() / data.num_nodes
        circuit_acc += ((circuit_out[:,1].flatten() > 0) == data.reachable).sum().item() / data.num_nodes
        circuit_ablated_acc += ((circuit_ablated_out[:,1].flatten() > 0) == data.reachable).sum().item() / data.num_nodes

    model_losses.append(model_loss)
    circuit_losses.append(circuit_loss)
    circuit_ablated_losses.append(circuit_ablated_loss)
    model_accs.append(model_acc)
    circuit_accs.append(circuit_acc)
    circuit_ablated_accs.append(circuit_ablated_acc)

In [12]:
print(f"{'seed':>5} {'model_loss':>10} {'circuit_loss':>12} {'ablated_loss':>12} {'model_acc':>10} {'circuit_acc':>12} {'ablated_acc':>12}")
for seed, model_loss, circuit_loss, ablated_loss, model_acc, circuit_acc, ablated_acc in zip(seeds, model_losses, circuit_losses, circuit_ablated_losses, model_accs, circuit_accs, circuit_ablated_accs):
    print(f"{seed:>5} {model_loss:>10.4f} {circuit_loss:>12.4f} {ablated_loss:>12.4f} {model_acc:>10.4f} {circuit_acc:>12.4f} {ablated_acc:>12.4f}")

print(f"\n{'metric':>12} {'mean':>12} {'std':>12}")
print(f"{'model_loss':>12} {torch.tensor(model_losses).mean().item():>12.4f} {torch.tensor(model_losses).std().item():>12.4f}")
print(f"{'circuit_loss':>12} {torch.tensor(circuit_losses).mean().item():>12.4f} {torch.tensor(circuit_losses).std().item():>12.4f}")
print(f"{'ablated_loss':>12} {torch.tensor(circuit_ablated_losses).mean().item():>12.4f} {torch.tensor(circuit_ablated_losses).std().item():>12.4f}")
print(f"{'model_acc':>12} {torch.tensor(model_accs).mean().item():>12.4f} {torch.tensor(model_accs).std().item():>12.4f}")
print(f"{'circuit_acc':>12} {torch.tensor(circuit_accs).mean().item():>12.4f} {torch.tensor(circuit_accs).std().item():>12.4f}")
print(f"{'ablated_acc':>12} {torch.tensor(circuit_ablated_accs).mean().item():>12.4f} {torch.tensor(circuit_ablated_accs).std().item():>12.4f}")

 seed model_loss circuit_loss ablated_loss  model_acc  circuit_acc  ablated_acc
    0     0.0519       0.0499   58038.1680     1.0000       0.9831       0.8161
    1     0.0604       0.0533   36308.5703     1.0000       0.9832       0.8161
    2     0.0519       0.0507    8855.6826     1.0000       0.9831       0.8161
    3     0.0499       0.0508   18173.6738     1.0000       0.9831       0.8161
    4     0.0602       0.0542   25983.5723     1.0000       0.9832       0.8161

      metric         mean          std
  model_loss       0.0549       0.0050
circuit_loss       0.0518       0.0019
ablated_loss   29471.9336   18891.9199
   model_acc       1.0000       0.0000
 circuit_acc       0.9831       0.0001
 ablated_acc       0.8161       0.0000


In [15]:
for seed, model_loss, circuit_loss, ablated_loss, model_acc, circuit_acc, ablated_acc in zip(seeds, model_losses, circuit_losses, circuit_ablated_losses, model_accs, circuit_accs, circuit_ablated_accs):
    print(f"{seed} & {model_loss:.4f} & {circuit_loss:.4f} & {ablated_loss:.4f} & {model_acc * 100:.2f} & {circuit_acc * 100:.2f} & {ablated_acc * 100:.2f} \\\\")

0 & 0.0519 & 0.0499 & 58038.1680 & 100.00 & 98.31 & 81.61 \\
1 & 0.0604 & 0.0533 & 36308.5703 & 100.00 & 98.32 & 81.61 \\
2 & 0.0519 & 0.0507 & 8855.6826 & 100.00 & 98.31 & 81.61 \\
3 & 0.0499 & 0.0508 & 18173.6738 & 100.00 & 98.31 & 81.61 \\
4 & 0.0602 & 0.0542 & 25983.5723 & 100.00 & 98.32 & 81.61 \\
